# Notebook 12 – Train, Validation & Test Data
## Training, Validation, and Test Datasets

Three different jobs, three different datasets:

* **Training set**: the data the model actually learns from.
* **Validation set**: used during development to tune settings (like model
  parameters) and compare different approaches, without touching the final
  test set.
* **Test set**: touched exactly once, at the very end, to report how the
  model performs on data it has never seen in any form.

If the model or any preprocessing step has "seen" the test set before that
final check, the test result stops meaning anything.

In [4]:
import pandas as pd
import glob
import os

# Automatically locate the CSV, even if it's not in the current working directory
matches = glob.glob("**/hr_employee_attrition_raw.csv", recursive=True)

if matches:
    file_path = matches[0]
    print("Found file at:", file_path)
else:
    raise FileNotFoundError(
        "Could not find 'hr_employee_attrition_raw.csv' anywhere under: "
        + os.getcwd()
        + "\nMake sure the file is saved somewhere inside this folder or a subfolder."
    )

df = pd.read_csv(file_path)
df["Attrition_binary"] = df["Attrition"].map({"Yes": 1, "No": 0})
df.shape

Found file at: Downloads\hr_employee_attrition_raw.csv


(1230, 17)

## Train-Test Split

The simplest split, one cut, two pieces.

In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

Train rows: 984
Test rows: 246


## Train-Validation-Test Split

For real projects, a two-way split isn't enough, you need a validation set
to tune things without ever touching the test set. This is usually done as
two splits back to back.

In [6]:
# First split off the test set, then split the remainder into train/validation
train_val_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=42)

print("Train:", len(train_df), "| Validation:", len(val_df), "| Test:", len(test_df))

Train: 738 | Validation: 246 | Test: 246


## Random State

**What it does:** fixes the "randomness" of the split so it's reproducible.
Running the same code with the same `random_state` gives the exact same
split every time, useful for debugging and fair comparisons between models.

Without it, every run creates a different split, making it impossible to
tell if a performance change came from your model or just a different
random split.

In [7]:
# Same random_state = same split, every time
split_1, _ = train_test_split(df, test_size=0.2, random_state=42)
split_2, _ = train_test_split(df, test_size=0.2, random_state=42)
print("Identical splits:", split_1.index.equals(split_2.index))

Identical splits: True


## Stratified Sampling

**The problem:** a plain random split can accidentally create an even more
imbalanced train/test split by chance, especially with a minority class as
small as ours (19% "Yes").

**The fix:** `stratify` forces both the train and test sets to keep the
same class proportions as the original data.

In [9]:
print("Original Attrition ratio:")
print(df["Attrition"].value_counts(normalize=True).round(3))

# Without stratify, class balance can drift between train and test by chance
train_plain, test_plain = train_test_split(df, test_size=0.2, random_state=42)
print("\nTest set ratio WITHOUT stratify:")
print(test_plain["Attrition"].value_counts(normalize=True).round(3))

# With stratify, both sets keep the same 81/19 split
train_strat, test_strat = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["Attrition"]
)
print("\nTest set ratio WITH stratify:")
print(test_strat["Attrition"].value_counts(normalize=True).round(3))

Original Attrition ratio:
Attrition
No     0.814
Yes    0.186
Name: proportion, dtype: float64

Test set ratio WITHOUT stratify:
Attrition
No     0.833
Yes    0.167
Name: proportion, dtype: float64

Test set ratio WITH stratify:
Attrition
No     0.813
Yes    0.187
Name: proportion, dtype: float64


## Data Leakage: Incorrect Workflow (Do Not Do This)

This is the mistake we already ran into back in Notebook 7, and it's worth
showing clearly as a "before" example.

Fitting a scaler or encoder on the **full dataset**, before splitting,
means the training process indirectly "sees" statistics from the test set
(its mean, its min/max, its category frequencies). The model then performs
better than it honestly should during evaluation, and that inflated number
won't hold up on real, truly unseen data.

In [10]:
# INCORRECT: fit_transform on the whole dataset BEFORE splitting
from sklearn.preprocessing import StandardScaler

scaler_wrong = StandardScaler()
df["MonthlyIncome_clean"] = pd.to_numeric(
    df["MonthlyIncome"].astype(str).str.replace(" USD", "", regex=False), errors="coerce"
)
df["Income_scaled_LEAKED"] = scaler_wrong.fit_transform(df[["MonthlyIncome_clean"]].fillna(0))

# The mean/std used below already includes rows that will end up in the test set
train_leaked, test_leaked = train_test_split(df, test_size=0.2, random_state=42)
print("This scaler's mean/std were computed using test rows too:", scaler_wrong.mean_)

This scaler's mean/std were computed using test rows too: [5498.18313781]


## Data Leakage: Correct Workflow

**The rule: split first, fit preprocessing only on the training set, then
apply that same fitted transformer to validation and test.**

The test set should never influence any statistic used during
preprocessing, not the mean, not the min/max, not category frequencies,
nothing.

In [11]:
# CORRECT: split FIRST
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["Attrition"])

# Fit the scaler ONLY on training data
scaler_correct = StandardScaler()
train_df = train_df.copy()
test_df = test_df.copy()

train_df["Income_scaled"] = scaler_correct.fit_transform(train_df[["MonthlyIncome_clean"]].fillna(0))

# Apply the SAME fitted scaler to test data, never re-fit it
test_df["Income_scaled"] = scaler_correct.transform(test_df[["MonthlyIncome_clean"]].fillna(0))

print("Mean used for scaling (from training data only):", scaler_correct.mean_)

Mean used for scaling (from training data only): [5369.52788129]


## Why This Applies to Every Preprocessing Step

This exact split-then-fit rule from Notebook 7's target encoding mistake
applies everywhere, not just scaling:

* **Scaling** (StandardScaler, MinMaxScaler): fit on train, apply to
  validation/test.
* **Encoding** (target encoding, frequency encoding): compute category
  statistics using only training rows.
* **Imputation** (filling missing values): compute the fill value (mean,
  median, mode) from training data only.
* **Outlier bounds** (IQR, z-score thresholds): calculate bounds from
  training data only, then apply those same bounds elsewhere.

If a step uses `.fit()` or `.fit_transform()`, that fit should only ever
happen on the training set.

In [18]:
role_attrition_rate = train_df.groupby("JobRole")["Attrition_binary"].mean()
train_df["JobRole_target_encoded"] = train_df["JobRole"].map(role_attrition_rate)
test_df["JobRole_target_encoded"] = test_df["JobRole"].map(role_attrition_rate)  # apply, don't refit
train_df[["JobRole", "JobRole_target_encoded"]].head()

,JobRole,JobRole_target_encoded
919,HR Manager,0.224719
393,Sales Executive,0.157143
427,HR Manager,0.224719
275,Support Lead,0.164557
907,Brand Manager,0.236842


## Temporal Splitting

**What it does:** instead of splitting randomly, splits by time, older
records go into training, newer records go into testing.

**When to use it:** whenever the data has a time dimension and the model
will be used to predict future outcomes. Our `JoinDate` column ranges from
2015 to 2025, a random split could put a 2024 hire in training and a 2015
hire in testing, which doesn't reflect how the model will actually be used
in practice (predicting forward in time, not backward).

In [19]:
df["JoinDate"] = pd.to_datetime(df["JoinDate"], errors="coerce")
df_sorted = df.sort_values("JoinDate")

cutoff_index = int(len(df_sorted) * 0.8)
train_temporal = df_sorted.iloc[:cutoff_index]
test_temporal = df_sorted.iloc[cutoff_index:]

print("Train date range:", train_temporal["JoinDate"].min(), "to", train_temporal["JoinDate"].max())
print("Test date range:", test_temporal["JoinDate"].min(), "to", test_temporal["JoinDate"].max())

Train date range: 2015-01-01 00:00:00 to 2023-02-24 00:00:00
Test date range: 2023-03-02 00:00:00 to 2025-05-26 00:00:00
